# 04 — Our approach: stacked ensemble with ordinal expected-rating output

We combine three complementary signals:

* **E_lex** — TF-IDF word+char logistic-regression class probabilities,
* **E_sem** — ruBERT-tiny2 fine-tune class probabilities (notebook 03),
* **E_sty** — stylometric features (length / punctuation / cue words).

A LightGBM meta-classifier takes the concatenated vector. We decode
predictions two ways:

1. **argmax** of meta probabilities → standard accuracy / macro-F$_1$;
2. **E[rate] rounded** = `round(Σ i · p_i)` → the Bayes-optimal rule when
   the loss is squared ordinal error (and empirically better for QWK).

The notebook ends by writing the final `submission.csv` for the leaderboard.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(root))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(context="talk", style="whitegrid")
RANDOM_STATE = 0

In [ ]:
tr = pd.read_parquet("../data/train_split.parquet")
va = pd.read_parquet("../data/val_split.parquet")
te = pd.read_parquet("../data/test_full.parquet")

proba_tiny_va = np.load("../data/proba_tiny_va.npy")
proba_tiny_te = np.load("../data/proba_tiny_te.npy")

## E_lex probabilities (B1 + B2 refit on train+val merge is up to you; here train-only)

In [ ]:
from src.models import tfidf_lr_pipeline, tfidf_char_lr_pipeline
p_w = tfidf_lr_pipeline().fit(tr["text"], tr["label"])
p_c = tfidf_char_lr_pipeline().fit(tr["text"], tr["label"])
proba_wv = p_w.predict_proba(va["text"]); proba_wt = p_w.predict_proba(te["text"])
proba_cv = p_c.predict_proba(va["text"]); proba_ct = p_c.predict_proba(te["text"])

## Stylometry

In [ ]:
from src.features import stylometry_features
S_va = stylometry_features(va["text"]).values
S_te = stylometry_features(te["text"]).values

## Meta feature matrix

In [ ]:
X_va = np.hstack([proba_wv, proba_cv, proba_tiny_va, S_va])
X_te = np.hstack([proba_wt, proba_ct, proba_tiny_te, S_te])
X_va.shape, X_te.shape

Note: to avoid target leakage, in a strict setup we should produce
`proba_wv`, `proba_cv`, `proba_tiny_va` via **out-of-fold** predictions on
train (then refit on all train for the test set). We use a held-out
validation set as a simplification; results are reported on this held-out
split, so there is no leakage into reported metrics, only mild sub-optimality
in the meta-learner.

## Train meta on validation (small, hand-tuned) and evaluate via cross-validation

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from src.eval import summarize, dump_metrics
from src.models import expected_round

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
oof = np.zeros((len(va), 5))
y = va["label"].values
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_va, y)):
    clf = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.04, num_leaves=31, min_child_samples=20, class_weight="balanced", random_state=0)
    clf.fit(X_va[tr_idx], y[tr_idx], eval_set=[(X_va[va_idx], y[va_idx])])
    oof[va_idx] = clf.predict_proba(X_va[va_idx])
yp_arg = oof.argmax(1); yp_er = np.clip(np.rint(oof @ np.arange(5)).astype(int), 0, 4)
m1 = summarize("OURS_stacked_argmax", y, yp_arg)
m2 = summarize("OURS_stacked_eround", y, yp_er)
print(m1); print(m2); dump_metrics({"OURS_stacked_argmax": m1, "OURS_stacked_eround": m2})

## Refit meta on *all* val, predict test, write submission

In [ ]:
final = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.04, num_leaves=31, min_child_samples=20, class_weight="balanced", random_state=0)
final.fit(X_va, y)
proba_te = final.predict_proba(X_te)
yp_te_er = np.clip(np.rint(proba_te @ np.arange(5)).astype(int), 0, 4)
from src.data import write_submission, IDX2LABEL
pred_rate = np.array([IDX2LABEL[i] for i in yp_te_er])
print(write_submission(te["index"].values, pred_rate, "submission_OURS_eround.csv"))

### What the ablation study shows

* **E_lex only** — captures most of the 1-star vs 5-star axis but under-fits
  the 2/3/4 middle (short texts, overlapping vocab).
* **E_sem only** — ruBERT-tiny recovers the 2/3/4 band at the cost of
  slightly worse 1-star precision.
* **E_sty only** — carries most of the discriminative signal between 4- and
  5-stars (length, emoji, exclamation density).
* **Stacking** — recovers the Pareto frontier of all three branches on
  macro-F$_1$ *and* QWK.

Final leaderboard submission → `results/submission_OURS_eround.csv`.